# Trump-Related Films — OMDb API Dataset Builder

Pulls metadata and scores (IMDb, Rotten Tomatoes, Metacritic) for films and documentaries related to Donald Trump using the [OMDb API](http://www.omdbapi.com/).

In [ ]:
import requests
import pandas as pd
import time

API_KEY = "YOUR_KEY_HERE"

import requests
import pandas as pd
import time

API_KEY = "64b1a1b2"

In [ ]:
# (title, year) tuples — year helps disambiguate
TRUMP_FILMS = [
    # Biopics / dramatizations
    ("The Apprentice", 2024),
    ("Bad President", 2021),
    
    # Documentaries — critical
    ("A Storm Foretold", 2023),
    ("Fahrenheit 11/9", 2018),
    ("#Unfit: The Psychology of Donald Trump", 2020),
    ("Get Me Roger Stone", 2017),
    ("You've Been Trumped", 2011),
    ("You've Been Trumped Too", 2016),
    ("Trumped: Inside the Greatest Political Upset of All Time", 2017),
    ("Looking for Melania Trump", 2020),
    ("A Dangerous Game", 2014),
    
    # Documentaries — pro-Trump / conservative
    ("Vindicating Trump", 2024),
    ("Trump Card", 2020),
    ("The Plot Against the President", 2020),
    ("My Dinner with Trump", 2022),
    ("Government Gangsters", 2024),
    
    # Documentaries — neutral / broader scope
    ("The First Step", 2021),
    ("Harry Benson: Shoot First", 2016),
    
    # TV miniseries
    ("The Comey Rule", 2020),
    ("Trump: An American Dream", 2017),
    
    # Older appearances
    ("Celebrity", 1998),
    ("Ghosts Can't Do It", 1990),
]

## Query OMDb

For each film, we hit the OMDb API with `tomatoes=true` to get extended Rotten Tomatoes data alongside IMDb and Metacritic scores.

In [ ]:
BASE_URL = "http://www.omdbapi.com/"

results = []

for title, year in TRUMP_FILMS:
    params = {
        "t": title,
        "y": year,
        "tomatoes": "true",
        "apikey": API_KEY,
    }
    resp = requests.get(BASE_URL, params=params)
    data = resp.json()
    
    if data.get("Response") == "False":
        print(f"NOT FOUND: {title} ({year}) — {data.get('Error')}")
        results.append({"Title": title, "Year": str(year), "Error": data.get("Error")})
        continue
    
    # Extract scores from the Ratings array
    ratings = {r["Source"]: r["Value"] for r in data.get("Ratings", [])}
    
    results.append({
        "Title": data.get("Title"),
        "Year": data.get("Year"),
        "Type": data.get("Type"),
        "Genre": data.get("Genre"),
        "Director": data.get("Director"),
        "Runtime": data.get("Runtime"),
        "Plot": data.get("Plot"),
        "imdbRating": data.get("imdbRating"),
        "imdbVotes": data.get("imdbVotes"),
        "imdbID": data.get("imdbID"),
        "RT_Tomatometer": ratings.get("Rotten Tomatoes"),
        "Metacritic": ratings.get("Metacritic"),
        "BoxOffice": data.get("BoxOffice"),
        "Rated": data.get("Rated"),
    })
    
    print(f"OK: {data.get('Title')} ({data.get('Year')})")
    time.sleep(0.25)  # stay well under rate limits

print(f"\nDone: {len(results)} films processed")

## Build DataFrame

In [ ]:
df = pd.DataFrame(results)

# Drop error-only rows if any, but keep a note
not_found = df[df["Error"].notna()] if "Error" in df.columns else pd.DataFrame()
if len(not_found) > 0:
    print("Films not found in OMDb:")
    print(not_found[["Title", "Year", "Error"]].to_string(index=False))
    print()

# Keep only successful results
if "Error" in df.columns:
    df = df[df["Error"].isna()].drop(columns=["Error"])

df

## Save to CSV

In [ ]:
df.to_csv("trump-films-omdb.csv", index=False)
print(f"Saved {len(df)} films to trump-films-omdb.csv")

## Quick Summary

In [ ]:
print(f"Total films: {len(df)}")
print(f"With RT score: {df['RT_Tomatometer'].notna().sum()}")
print(f"With Metacritic score: {df['Metacritic'].notna().sum()}")
print(f"With IMDb rating: {df['imdbRating'].notna().sum()}")
print()
print("Genre breakdown:")
print(df["Genre"].value_counts().to_string())